In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

import copy

### Model and Dataset definition

In [ ]:
class UNSW_NB15_Dataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.Y)
    
    def __getitem__(self, index):
        sample = self.X[index]
        label = self.Y[index]
        return sample, label

In [ ]:
# Define the neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size=2):
        super(SimpleNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_layers[0], bias=False),
            nn.BatchNorm1d(hidden_layers[0]),
            nn.ReLU(),
            nn.Linear(hidden_layers[0], hidden_layers[1], bias=False),
            nn.BatchNorm1d(hidden_layers[1]),
            nn.ReLU(),
            nn.Linear(hidden_layers[1], hidden_layers[2], bias=False),
            nn.BatchNorm1d(hidden_layers[2]),
            nn.ReLU(),
            nn.Linear(hidden_layers[2], output_size, bias=False),
        )

    def forward(self, x):
        return self.model(x)

### Data import and cleaning 

In [ ]:
data_tr = pd.read_csv('../datasets/UNSW_NB15_training-set.csv', delimiter=',')
data_tst = pd.read_csv('../datasets/UNSW_NB15_testing-set.csv', delimiter=',')

In [ ]:
# data preparation
categorical_features_values = 6
continuous_features_values  = 50
list_drop = ['id','attack_cat']

data = pd.concat([data_tr])
data.drop(list_drop,axis=1,inplace=True)

# clamping continuous values
data_num = data.select_dtypes(include=[np.number])
for feature in data_num.columns:
    if data_num[feature].max()>10*data_num[feature].median() and data_num[feature].max()>10 :
        data[feature] = np.where(data[feature]<data[feature].quantile(0.95), data[feature], data[feature].quantile(0.95))

# unskew data applying log
data_num = data.select_dtypes(include=[np.number])
for feature in data_num.columns:
    if data_num[feature].nunique()>continuous_features_values:
        if data_num[feature].min()==0:
            data[feature] = np.log(data[feature]+1)
        else:
            data[feature] = np.log(data[feature])

# limit non-unique value for categorical features
data_cat = data.select_dtypes(exclude=[np.number])
for feature in data_cat.columns:
    if data_cat[feature].nunique()>categorical_features_values:
        data[feature] = np.where(data[feature].isin(data[feature].value_counts().head(categorical_features_values).index), data[feature], '-')

# split features and labels
samples = data[data.columns[:-1]]
labels = data[data.columns[-1]]

# one hot encoding of categorical features
print(f'Feature count before one-hot encoding: {samples.shape[1]}')
one_hot = pd.get_dummies(data=samples, columns=data_cat.columns)
old_samples = samples
print(f'Feature count after one-hot encoding: {one_hot.shape[1]}')
print(f'Added   features: {one_hot[one_hot.columns.difference(old_samples.columns)].columns.to_list()}')
print(f'Removed features: {old_samples[old_samples.columns.difference(one_hot.columns)].columns.to_list()}')
samples=one_hot

# from bool features to int
bool_cols = samples.select_dtypes(include=bool).columns
samples[bool_cols] = samples[bool_cols].astype(int)

# normalize float column
scaler = StandardScaler()
data_num = samples.select_dtypes(include=np.number)
samples[data_num.columns] = scaler.fit_transform(data_num)

X_train = torch.tensor(samples.values, dtype=torch.float32)
Y_train = torch.tensor(labels.values, dtype=torch.long)

In [ ]:
def train_loop(train_dataloader: DataLoader, model: SimpleNN, lr, epochs, patience, X_val=None, Y_val=None,fold_idx=None, dummy_size=None):
    size = len(train_dataloader.dataset)
    

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr)
    if dummy_size == None:
        scheduler = ReduceLROnPlateau(optimizer=optimizer, patience=patience, verbose=True, min_lr=1e-5)

    best_acc = -np.inf
    best_weights = None
    
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")

        model.train()
        for batch, (X, Y) in enumerate(train_dataloader):
            if dummy_size != None and batch >= dummy_size:
                break
            
            # forward pass
            pred = model(X)
            loss = loss_fn(pred, Y)
            # backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # training stats every x batches
            if(batch % 50 == 0):
                loss, current = loss.item(), batch * train_dataloader.batch_size + len(X)
                acc = (pred.argmax(1).round() == Y).float().mean()
                print(f"loss: {loss:>7f} acc: {acc*100}% [{current:>5d}/{size:>5d}]")

        if(X_val != None and Y_val != None):
            model.eval()
            pred = model(X_val)
            val_loss = loss_fn(pred, Y_val)
            acc = (pred.argmax(1).round() == Y_val).float().mean()
            print(f'evaluation fold {fold_idx} acc: {acc} ')
            if acc > best_acc:
                best_acc = acc
                best_weights = copy.deepcopy(model.state_dict())
        
        # lr decay
        if dummy_size == None:
            scheduler.step(val_loss)  
    
    return best_weights, best_acc
        
    

#### Hyper parameter definition

In [ ]:
batch_size = 1024
epochs = 100
lr = 1e-2
kfold_splits = 5
kfold_idx = 0
patience = 5
hidden_layers = [32, 7, 9]
input_neuron_size = 59

### Train and validation phase

In [ ]:
sss = StratifiedShuffleSplit(n_splits=kfold_splits, random_state=0, test_size=0.1)
best_model = None
best_acc = -np.inf
for train_ids, val_ids in sss.split(X_train, Y_train):
    model = SimpleNN(hidden_layers=hidden_layers, input_size=input_neuron_size)
    train_dataset = UNSW_NB15_Dataset(X_train[train_ids], Y_train[train_ids])
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
    X_val, Y_val = X_train[val_ids], Y_train[val_ids]
    model, acc = train_loop(train_dataloader, model, lr, epochs, patience, X_val, Y_val, kfold_idx)
    
    if(acc > best_acc):
        best_acc=acc
    print(56*'*')
    print(f'*FOLD N {kfold_idx} ENDED - BEST ACC SO FAR: {best_acc}*')
    print(56*'*')
    kfold_idx += 1


torch.save(best_model, f'models/bnn_acc_{best_acc:.3f}.pth')